Task 1: News Topic Classifier (BERT)


In [2]:
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

# 1. Load Dataset
dataset = load_dataset("ag_news")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# 2. Tokenize
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 3. Load Model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4)

# 4. Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    per_device_train_batch_size=8,
    num_train_epochs=1
)

# 5. Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].shuffle(seed=42).select(range(1000)), # Sample for speed
    eval_dataset=tokenized_datasets["test"].shuffle(seed=42).select(range(500))
)

trainer.train()

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,No log,0.441144


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=125, training_loss=0.6371664428710937, metrics={'train_runtime': 126.3734, 'train_samples_per_second': 7.913, 'train_steps_per_second': 0.989, 'total_flos': 263115780096000.0, 'train_loss': 0.6371664428710937, 'epoch': 1.0})

Task 2: End-to-End ML Pipeline (Scikit-learn)

In [5]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import kagglehub
import os # Import the os module

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Path to dataset files:", path)
# 1. Load Dataset (Assuming local csv or URL)
df = pd.read_csv(os.path.join(path, 'WA_Fn-UseC_-Telco-Customer-Churn.csv')) # Corrected file path
X = df.drop('Churn', axis=1)
y = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

# 2. Define Preprocessing [cite: 29]
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# 3. Create Pipeline [cite: 29, 30]
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])

# 4. Hyperparameter Tuning [cite: 31]
param_grid = {'classifier__n_estimators': [50, 100], 'classifier__max_depth': [None, 10]}
grid_search = GridSearchCV(pipeline, param_grid, cv=5)
grid_search.fit(X, y)

# 5. Export Pipeline [cite: 32]
joblib.dump(grid_search.best_estimator_, 'churn_pipeline.pkl')
print("Pipeline saved successfully.")

Using Colab cache for faster access to the 'telco-customer-churn' dataset.
Path to dataset files: /kaggle/input/telco-customer-churn
Pipeline saved successfully.


Task 5: Auto Tagging Support Tickets (LLM)

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

# 1. Setup Prompt Template (re-using the previous template as a simple string)
template = """
You are a support ticket classifier. Tag the following ticket into the top 3 categories.
Categories: Technical, Billing, General Inquiry, Feature Request, Account Access.

Ticket: {ticket_text}

Output only the top 3 tags:
"""

# 2. Load a local LLM (e.g., distilgpt2) via HuggingFace pipeline
# You can choose a different model if you have more resources
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Create a Hugging Face pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=50, # Limit the output length for classification tags
    temperature=0.1,   # Make the output more deterministic
    top_p=0.95,
    repetition_penalty=1.1,
    device=0 if torch.cuda.is_available() else -1 # Use GPU if available
)

# 3. Few-shot logic / Zero-shot (direct interaction with Hugging Face pipeline)
def tag_ticket_local_llm(text):
    formatted_prompt = template.format(ticket_text=text)
    print(f"Processing ticket with prompt: {formatted_prompt}")

    # Generate text using the Hugging Face pipeline
    # The pipeline will return a list of dictionaries
    outputs = pipe(formatted_prompt, max_new_tokens=50, num_return_sequences=1)
    generated_text = outputs[0]['generated_text']

    # Post-process the generated text to extract the response part
    # distilgpt2 is a small model and might not follow instructions perfectly,
    # so this is a basic extraction. You might need more robust parsing
    # for stronger models or specific output formats.
    start_index = len(formatted_prompt)
    extracted_tags = generated_text[start_index:].strip()

    # Attempt to clean up common distilgpt2 generation artifacts (optional, heuristic)
    if extracted_tags and extracted_tags[0].islower():
        # Try to find the start of a logical tag or sentence after the prompt
        first_upper = next((i for i, char in enumerate(extracted_tags) if char.isupper()), -1)
        if first_upper != -1:
            extracted_tags = extracted_tags[first_upper:]

    return extracted_tags

# 4. Example Execution
sample_ticket = "My internet connection is constantly dropping, and I can't access any websites."
tags = tag_ticket_local_llm(sample_ticket)
print(f"\nGenerated Tags: {tags}")

sample_ticket_2 = "I was charged twice for my last subscription renewal."
tags_2 = tag_ticket_local_llm(sample_ticket_2)
print(f"\nGenerated Tags: {tags_2}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'repetition_penalty', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing ticket with prompt: 
You are a support ticket classifier. Tag the following ticket into the top 3 categories.
Categories: Technical, Billing, General Inquiry, Feature Request, Account Access.

Ticket: My internet connection is constantly dropping, and I can't access any websites.

Output only the top 3 tags:



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Tags: 1) The first tag will be displayed in your browser (i.e., if you have an account with no username or password).
2)(3)- You may not use this feature on other sites unless it's enabled by default.
4
Processing ticket with prompt: 
You are a support ticket classifier. Tag the following ticket into the top 3 categories.
Categories: Technical, Billing, General Inquiry, Feature Request, Account Access.

Ticket: I was charged twice for my last subscription renewal.

Output only the top 3 tags:


Generated Tags: I am not sure if this is true or false but it's possible that some of you may have noticed something wrong with your account and/or email address (which might be incorrect). Please contact me to see how much time has been spent on fixing
